In [3]:
from ultralytics import YOLO
import yaml

YAML_PATH = "final_dataset_yoloOnly/dataset_yolo.yaml"
MODEL_CFG = "yolov8n-p2.yaml"

def create_p2_config():
    """
    Стандарт: P3(8), P4(16), P5(32).
    Цей конфіг: P2(4), P3(8), P4(16), P5(32).
    Це критично для дрібних цифр.
    """
    p2_yaml = {
        'nc': 13,
        'scales': {
            'n': [0.33, 0.25, 1024]
        },
        'backbone': [
            [-1, 1, 'Conv', [64, 3, 2]],  # 0-P1/2
            [-1, 1, 'Conv', [128, 3, 2]], # 1-P2/4
            [-1, 3, 'C2f', [128, True]],
            [-1, 1, 'Conv', [256, 3, 2]], # 3-P3/8
            [-1, 6, 'C2f', [256, True]],
            [-1, 1, 'Conv', [512, 3, 2]], # 5-P4/16
            [-1, 6, 'C2f', [512, True]],
            [-1, 1, 'Conv', [1024, 3, 2]], # 7-P5/32
            [-1, 3, 'C2f', [1024, True]],
            [-1, 1, 'SPPF', [1024, 5]]  # 9
        ],
        'head': [
            [-1, 1, 'nn.Upsample', [None, 2, 'nearest']],
            [[-1, 6], 1, 'Concat', [1]],  # cat backbone P4
            [-1, 3, 'C2f', [512]],  # 12

            [-1, 1, 'nn.Upsample', [None, 2, 'nearest']],
            [[-1, 4], 1, 'Concat', [1]],  # cat backbone P3
            [-1, 3, 'C2f', [256]],  # 15 (P3/8-small)

            [-1, 1, 'nn.Upsample', [None, 2, 'nearest']],
            [[-1, 2], 1, 'Concat', [1]],  # cat backbone P2
            [-1, 3, 'C2f', [128]],  # 18 (P2/4-xsmall)

            [[18, 15, 12, 9], 1, 'Detect', ['nc']]  # Detect(P2, P3, P4, P5)
        ]
    }

    with open(MODEL_CFG, 'w') as f:
        yaml.dump(p2_yaml, f, sort_keys=False)
    print(f"Створено конфігурацію {MODEL_CFG} з шаром P2")

def main():
    create_p2_config()
    model = YOLO(MODEL_CFG)

    try:
        model.load("yolov8n.pt")
        print("Ваги Backbone завантажено з yolov8n.pt")
    except:
        print("Тренування з нуля (ваги не підтягнулись, це норм для кастомної архітектури)")

    print("Починаємо тренування YOLO-P2 (High Res) на 13 класів...")

    results = model.train(
        data=YAML_PATH,
        epochs=30,
        imgsz=640,
        batch=16,
        name="yolo_only",
        patience=8,
        device='0',
        verbose=True,

        fliplr=0.0,
        flipud=0.0,
        degrees=5.0,
        mosaic=1.0,
        scale=0.5,
    )

    print("Тренування завершено!")
    print(f"Найкраща модель тут: runs/detect/yolo_only/weights/best.pt")

if __name__ == "__main__":
    main()

✅ Створено конфігурацію yolov8n-p2.yaml з шаром P2
Transferred 219/437 items from pretrained weights
✅ Ваги Backbone завантажено з yolov8n.pt
🚀 Починаємо тренування YOLO-P2 (High Res) на 13 класів...
New https://pypi.org/project/ultralytics/8.3.233 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.224 🚀 Python-3.12.3 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=final_dataset_yoloOnly/dataset_yolo.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8